# 17 — Recomendação de ação para revisão humana

Este módulo não executa ações sobre clientes. Ele converte os indicadores em uma label de próximo passo e marca `revisao_humana=True` em todos os casos.


## Preparação para execução isolada

Esta célula carrega os fundamentos do notebook 12 quando este módulo é aberto sozinho. O notebook 18 já carrega os módulos na ordem necessária.


In [ ]:
if '_project_root' not in globals():
    from pathlib import Path
    _module_root = Path.cwd().resolve()
    if not (_module_root / '12_fundamentos_analise.ipynb').is_file():
        _module_root = _module_root / 'notebooks'
    if not (_module_root / '12_fundamentos_analise.ipynb').is_file():
        raise FileNotFoundError('Notebook 12 não encontrado na raiz ou em notebooks/.')
    get_ipython().run_line_magic('run', f'"{_module_root / "12_fundamentos_analise.ipynb"}"')


## Confiança comparável somente dentro de cada mecanismo

Como scores de modelo e heurística têm naturezas diferentes, o limiar serve apenas para direcionar casos frágeis à revisão manual; ele não é apresentado como probabilidade calibrada.

O limiar de 0,55 encaminha evidências fracas para revisão. Scores de produtos, modelos e regras têm significados diferentes e não devem ser comparados como probabilidades iguais.


In [ ]:
def _has_low_confidence(
    products: list[dict[str, Any]],
    sentiment: dict[str, Any],
    churn: dict[str, Any],
    opportunity: dict[str, Any],
) -> bool:
    indicators = (sentiment, churn, opportunity)
    if any(
        indicator["score_type"] == "model_probability"
        and float(indicator["score"]) < 0.55
        for indicator in indicators
    ):
        return True

    active_heuristic_scores = []
    if sentiment["score_type"] == "heuristic" and sentiment["label"] not in {"neutro", "misto"}:
        active_heuristic_scores.append(float(sentiment["score"]))
    if churn["score_type"] == "heuristic" and churn["label"] != "baixo":
        active_heuristic_scores.append(float(churn["score"]))
    if opportunity["score_type"] == "heuristic" and opportunity["label"] == "detectada":
        active_heuristic_scores.append(float(opportunity["score"]))
    if any(score < 0.55 for score in active_heuristic_scores):
        return True

    return bool(
        opportunity["label"] == "detectada"
        and products
        and float(products[0]["score"]) < 0.65
    )


## Ordem de prioridade

Churn alto aciona retenção. Depois vêm revisão manual, demonstração com produto explicitamente fundamentado, qualificação sem produto confiável e, por fim, acompanhamento da conta.

A função devolve `label`, `criterios`, `motivo` e `revisao_humana`. Os critérios são códigos estáveis associados aos indicadores usados; o motivo é uma leitura curta desses critérios. A demonstração só aparece quando há oportunidade, produto explícito, fonte e score suficiente. Mesmo assim, o nome do produto e a fonte precisam ser conferidos por uma pessoa, pois uma comparação pode aparecer antes do produto citado.


In [ ]:
def _recommend_action(
    products: list[dict[str, Any]],
    sentiment: dict[str, Any],
    churn: dict[str, Any],
    opportunity: dict[str, Any],
) -> dict[str, Any]:
    if churn["label"] == "alto":
        label = "acionar_retencao"
        criteria = ["risco_churn_alto"]
        reason = "O indicador de risco de churn está alto."
    elif sentiment["label"] == "misto":
        label = "revisar_manualmente"
        criteria = ["sentimento_misto"]
        reason = "O sentimento reúne sinais conflitantes."
    elif _has_low_confidence(products, sentiment, churn, opportunity):
        label = "revisar_manualmente"
        criteria = ["baixa_confianca"]
        reason = "Um indicador ativo ou produto candidato ficou abaixo do limiar de revisão."
    elif opportunity["label"] == "detectada":
        product_is_grounded = bool(
            products
            and products[0]["score"] >= 0.65
            and products[0]["explicit_match"]
            and products[0]["sources"]
        )
        criteria = ["oportunidade_detectada"]
        if product_is_grounded:
            label = "agendar_demonstracao"
            criteria.append("produto_explicito_com_fonte")
            reason = "Há oportunidade detectada e produto candidato explícito com fonte."
        else:
            label = "qualificar_oportunidade"
            criteria.append("produto_sem_fundamentacao_suficiente")
            reason = "Há oportunidade detectada, mas o produto candidato precisa de qualificação."
    else:
        label = "acompanhar_conta"
        criteria = ["oportunidade_nao_detectada"]
        reason = "Não foi detectada oportunidade comercial nesta transcrição."
    return {
        "label": label,
        "criterios": criteria,
        "motivo": reason,
        "revisao_humana": True,
    }
